# Dexter ACT 학습 (Colab, 무료 T4 GPU)

사용법과 체크리스트는 저장소의 `docs/10-colab.md`에 있습니다. **위에서부터 셀을 하나씩 실행**하세요.

1. 설정 → 2. GPU 확인 → 3. 설치 → 4. 데이터셋 올리기 → 5. 영상 디코딩 확인 → 6. 100스텝 시험 → 7. 본 학습 → 8. 체크포인트 내려받기

- 이 노트북은 **학습만** 합니다(평가는 서버에서). mujoco는 설치하지 않습니다.
- Hugging Face Hub와 wandb는 쓰지 않습니다(`--policy.push_to_hub=false`, `--wandb.enable=false`).
- 이 노트북은 Colab에서 실제로 실행해 보지 못한 상태로 작성됐습니다. 막히면 출력 메시지를 그대로 복사해서 알려 주세요.

In [ ]:
# ===== 1. 설정 (바꿀 것은 이 셀뿐) =====
DRIVE_DIR = '/content/drive/MyDrive/dexter'          # Drive 안의 작업 폴더(체크포인트가 여기에 저장됨)
DATASET_TAR = f'{DRIVE_DIR}/dexter_dataset_v1.tar'   # 서버에서 만들어 Drive에 올린 파일
REPO_ID = 'local/dexter_pick_place_v1'
DATASET_ROOT = '/content/data/dexter_pick_place_v1'
OUT_DIR = '/content/out'

STEPS = 20_000        # 총 학습 스텝(첫 시험은 20,000)
BATCH_SIZE = 8
SAVE_FREQ = 2_000     # 이 스텝마다 체크포인트 저장(세션이 끊겨도 여기서부터 이어할 수 있음)
LOG_FREQ = 100
NUM_WORKERS = 2
TRIAL_STEPS = 100     # 시험 학습 스텝
RESUME = False        # 세션이 끊긴 뒤 이어서 학습하려면 True (docs/10-colab.md 7번)
print('설정 완료')

In [ ]:
# ===== 2. GPU와 Python 확인 =====
import subprocess, sys
print(sys.version)
assert sys.version_info >= (3, 12), 'lerobot 0.6.1은 Python 3.12 이상이 필요합니다.'
try:
    print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
except FileNotFoundError:
    raise SystemExit('GPU가 없습니다. 메뉴 [런타임] > [런타임 유형 변경]에서 T4 GPU를 고르고 다시 시작하세요.')

In [ ]:
# ===== 3. lerobot 설치 (버전 고정: train/requirements.txt와 같은 0.6.1) =====
# Colab에 깔린 CUDA용 torch는 그대로 씁니다. 설치 중 numpy 관련 메시지가 나오면 [런타임] > [세션 다시 시작] 뒤 이 셀부터 다시 실행하세요.
!pip install -q 'lerobot[training]==0.6.1'

In [ ]:
import importlib.metadata as md
import torch
print('lerobot', md.version('lerobot'), '| torch', torch.__version__, '| numpy', md.version('numpy'), '| CUDA 사용 가능:', torch.cuda.is_available())
assert md.version('lerobot') == '0.6.1', 'lerobot 버전이 0.6.1이 아닙니다'
assert torch.cuda.is_available(), 'torch가 GPU를 못 봅니다(런타임 유형 확인)'

In [ ]:
# ===== 4. Drive 연결과 데이터셋 풀기 =====
import os, tarfile
from google.colab import drive
drive.mount('/content/drive')
assert os.path.exists(DATASET_TAR), f'{DATASET_TAR} 가 없습니다. docs/10-colab.md 3번(Drive에 올리기)을 확인하세요.'
os.makedirs('/content/data', exist_ok=True)
if not os.path.exists(DATASET_ROOT):
    with tarfile.open(DATASET_TAR) as tar:
        tar.extractall('/content/data')
print(sorted(os.listdir(DATASET_ROOT)), '(meta, data, videos 가 보여야 합니다)')

In [ ]:
# ===== 도우미 함수 (실행만 하면 됩니다) =====
import os, re, shutil, subprocess, threading, time
from pathlib import Path
import numpy as np


def summarize_trial(log_text, gpu_csv_text, total_steps, skip_first=10):
    """시험 학습 로그에서 한 줄 요약을 만든다: 스텝당 시간(updt_s + data_s), 총 스텝 예상 시간, GPU 메모리 최대."""
    upd = [float(x) for x in re.findall(r'updt_s:([0-9.]+)', log_text)]
    dat = [float(x) for x in re.findall(r'data_s:([0-9.]+)', log_text)]
    n = min(len(upd), len(dat))
    if n == 0:
        return '[요약] 로그에서 스텝 시간을 찾지 못했습니다. /content/trial.log 를 확인하세요.'
    start = skip_first if n > skip_first + 4 else 0
    per_step = float(np.mean(np.array(upd[start:n]) + np.array(dat[start:n])))
    mem_mib = [float(x) for x in re.findall(r'^[ ]*([0-9]+(?:[.][0-9]+)?)[ ]*MiB', gpu_csv_text, flags=re.M)]
    mem_text = f'{max(mem_mib) / 1024:.1f} GB' if mem_mib else '측정 못 함'
    hours = per_step * total_steps / 3600
    return f'[요약] 스텝당 {per_step:.2f}초, {total_steps:,}스텝 예상 {hours:.1f}시간, GPU 메모리 최대 {mem_text} (측정 스텝 {start + 1}~{n}번 평균)'


def run_train(output_dir, steps, log_path, log_freq, save_freq, resume_config=None):
    """lerobot-train 실행. 로그는 파일에 모두 쓰고 화면에는 step 줄과 오류만 보여 준다."""
    if resume_config:
        cmd = ['lerobot-train', '--resume=true', f'--config_path={resume_config}']
    else:
        cmd = ['lerobot-train', f'--dataset.repo_id={REPO_ID}', f'--dataset.root={DATASET_ROOT}', '--policy.type=act',
               '--policy.device=cuda', '--policy.push_to_hub=false', f'--batch_size={BATCH_SIZE}', f'--num_workers={NUM_WORKERS}',
               f'--steps={steps}', f'--log_freq={log_freq}', f'--save_freq={save_freq}', '--wandb.enable=false', f'--output_dir={output_dir}']
    env = dict(os.environ, HF_HUB_OFFLINE='1', WANDB_MODE='disabled', HF_HUB_DISABLE_TELEMETRY='1')
    print(' '.join(cmd))
    with open(log_path, 'a') as log:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, bufsize=1)
        for line in proc.stdout:
            log.write(line)
            if 'step:' in line:
                print(line[line.index('step:'):].rstrip()[:200])
            elif 'Traceback' in line or 'Error' in line:
                print(line.rstrip()[:300])
        code = proc.wait()
    if code != 0:
        raise RuntimeError(f'lerobot-train 이 오류로 끝났습니다(종료 코드 {code}). 위 출력과 {log_path} 를 확인하세요.')


def sync_checkpoints(src_out, dst_dir):
    """완성된 checkpoints/last 를 Drive로 복사한다. 10,000스텝 배수 체크포인트는 따로 남긴다. 이미 복사한 것은 건너뛴다."""
    ck = Path(src_out) / 'checkpoints'
    last = ck / 'last'
    if not last.exists():
        return None
    last_step = last.resolve().name
    dst_dir = Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)
    wanted = {'last': last.resolve()}
    for p in ck.iterdir():
        if p.is_dir() and not p.is_symlink() and p.name.isdigit() and int(p.name) % 10000 == 0 and int(p.name) <= int(last_step):
            wanted['step_' + p.name] = p
    for name, src in wanted.items():
        dst, marker = dst_dir / name, dst_dir / (name + '.step')
        if dst.exists() and marker.exists() and marker.read_text() == src.name:
            continue
        tmp = dst_dir / (name + '.tmp')
        shutil.rmtree(tmp, ignore_errors=True)
        shutil.copytree(src, tmp)
        shutil.rmtree(dst, ignore_errors=True)
        tmp.rename(dst)
        marker.write_text(src.name)
    return last_step


def restore_from_drive():
    """세션이 끊긴 뒤: Drive의 last 체크포인트를 /content/out 으로 되돌리고 이어서 학습할 설정 파일 경로를 돌려준다."""
    ck = Path(DRIVE_DIR) / 'checkpoints'
    step = (ck / 'last.step').read_text().strip()
    dst = Path(OUT_DIR) / 'checkpoints' / step
    if not dst.exists():
        shutil.copytree(ck / 'last', dst)
    link = Path(OUT_DIR) / 'checkpoints' / 'last'
    if link.is_symlink() or link.exists():
        link.unlink()
    link.symlink_to(step)
    return str(dst / 'pretrained_model' / 'train_config.json')


print('도우미 함수 준비 완료')

In [ ]:
# ===== 5. 영상 디코딩 확인 (AV1 4:4:4 영상이 이 Colab에서 열리는지) =====
os.environ['HF_HUB_OFFLINE'] = '1'
from lerobot.datasets import LeRobotDataset
try:
    ds = LeRobotDataset(REPO_ID, root=DATASET_ROOT)
    print('에피소드', ds.num_episodes, '프레임', ds.num_frames, '(기대값: 20, 13667)')
    ep = np.asarray(ds.hf_dataset.with_format('numpy')['episode_index']).reshape(-1)
    for e in (0, 7, 15, 19):
        i = int(np.argmax(ep == e)) + 5
        img = ds[i]['observation.images.main']
        assert tuple(img.shape) == (3, 240, 320), img.shape
        mean, std = img.mean().item(), img.std().item()
        assert 0.3 < mean < 0.95 and std > 0.02, f'이미지가 비정상입니다(평균 {mean:.3f}, 표준편차 {std:.3f})'
        print(f'에피소드 {e}, 프레임 {i}: shape {tuple(img.shape)}, 평균 {mean:.3f}, 표준편차 {std:.3f}')
    print('디코딩 확인 통과')
except Exception as err:
    print('디코딩 확인 실패:', type(err).__name__, err)
    print('대안: docs/10-colab.md의 "디코딩이 안 될 때"를 따라 서버에서 build_dataset.py --image-mode(또는 --codec h264)로 다시 만들어 올리세요.')
    raise

In [ ]:
# ===== 6. 100스텝 시험 학습: 스텝 시간과 GPU 메모리를 잰다 =====
shutil.rmtree('/content/trial_out', ignore_errors=True)
for f in ('/content/trial.log', '/content/gpu.csv'):
    Path(f).unlink(missing_ok=True)
gpu = subprocess.Popen(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader', '-l', '2'], stdout=open('/content/gpu.csv', 'w'))
t0 = time.time()
try:
    run_train('/content/trial_out', TRIAL_STEPS, '/content/trial.log', 1, TRIAL_STEPS + 1)
finally:
    gpu.terminate()
print()
print(summarize_trial(Path('/content/trial.log').read_text(), Path('/content/gpu.csv').read_text(), STEPS))
print(f'(위 [요약] 줄을 복사해서 보내 주세요. 시험 전체 {time.time() - t0:.0f}초)')

In [ ]:
# ===== 7. 본 학습 (체크포인트는 2분마다 Drive로 자동 복사) =====
Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)
resume_config = None
if RESUME:
    resume_config = restore_from_drive()
    print('이어서 학습:', resume_config)
elif Path(OUT_DIR).exists():
    raise SystemExit(f'{OUT_DIR} 가 이미 있습니다. 새로 시작하려면 폴더를 지우고, 이어서 하려면 RESUME = True 로 바꾸세요.')

stop = threading.Event()


def copier():
    while not stop.is_set():
        try:
            sync_checkpoints(OUT_DIR, f'{DRIVE_DIR}/checkpoints')
        except Exception as err:
            print('Drive 복사 실패(다음에 다시 시도):', err)
        stop.wait(120)


thread = threading.Thread(target=copier, daemon=True)
thread.start()
t0 = time.time()
try:
    run_train(OUT_DIR, STEPS, '/content/train.log', LOG_FREQ, SAVE_FREQ, resume_config)
finally:
    stop.set()
    thread.join(timeout=600)
    print('마지막 복사한 체크포인트 스텝:', sync_checkpoints(OUT_DIR, f'{DRIVE_DIR}/checkpoints'))
    shutil.copy('/content/train.log', f'{DRIVE_DIR}/train.log')
print(f'학습 끝: {(time.time() - t0) / 3600:.2f}시간')

In [ ]:
# ===== 8. 결과 확인과 내려받기 =====
from google.colab import files
model_dir = Path(DRIVE_DIR) / 'checkpoints' / 'last' / 'pretrained_model'
step = (Path(DRIVE_DIR) / 'checkpoints' / 'last.step').read_text().strip()
total = sum(f.stat().st_size for f in model_dir.rglob('*') if f.is_file())
print(f'체크포인트 스텝 {step}, pretrained_model 크기 {total / 1e6:.0f} MB (약 198 MB가 정상)')
for f in sorted(model_dir.iterdir()):
    print(f'  {f.name}  {f.stat().st_size / 1e6:.1f} MB')
tar_path = '/content/dexter_act_v1_pretrained_model.tar'
subprocess.run(['tar', 'cf', tar_path, '-C', str(model_dir.parent), 'pretrained_model'], check=True)
print(f'내려받을 파일: {tar_path} ({Path(tar_path).stat().st_size / 1e6:.0f} MB). Drive에도 {model_dir} 로 남아 있습니다.')
files.download(tar_path)

## 세션이 끊겼을 때 / 서버로 옮기기

- 끊기면 **1~4번 셀을 다시 실행**한 뒤 `RESUME = True`로 바꾸고 7번 셀부터 실행하세요(Drive의 마지막 체크포인트에서 이어집니다).
- 내려받은 `pretrained_model` 폴더를 서버의 `checkpoints/dexter_act_v1/` 아래로 옮기는 방법은 `docs/10-colab.md`를 보세요.